# Decision trees & random forests

Compute Gini gain, then inspect a tiny classification stump and a bootstrapped forest. Only the standard library is required. Synthetic training results are not a generalization claim.

In [1]:
from collections import Counter
from random import Random

def gini(labels):
    if not labels: return 0.0
    counts = Counter(labels)
    return 1 - sum((count/len(labels))**2 for count in counts.values())

def gain(parent, left, right):
    n = len(parent)
    return gini(parent) - (len(left)/n*gini(left) + len(right)/n*gini(right))

parent, left, right = [0,0,0,1,1,1], [0,0,0,1], [1,1]
print('parent, left, right Gini:', *[round(gini(v), 3) for v in (parent,left,right)])
print('weighted gain:', round(gain(parent,left,right), 3))

parent, left, right Gini: 0.5 0.375 0.0
weighted gain: 0.25


## Fit a one-split tree

This stump searches numeric thresholds on one feature. Ties are resolved deterministically. A full tree repeats this process within each resulting node.

In [2]:
def fit_stump(xs, ys):
    thresholds = [(a+b)/2 for a,b in zip(sorted(set(xs)), sorted(set(xs))[1:])]
    candidates = []
    for threshold in thresholds:
        left = [y for x,y in zip(xs,ys) if x <= threshold]
        right = [y for x,y in zip(xs,ys) if x > threshold]
        candidates.append((gain(ys,left,right), threshold, Counter(left).most_common(1)[0][0],
                           Counter(right).most_common(1)[0][0]))
    return max(candidates, key=lambda row: (row[0], -row[1]))

def predict(stump, x):
    _, threshold, left_label, right_label = stump
    return left_label if x <= threshold else right_label

xs, ys = [1,2,3,4,5,6], [0,0,0,1,1,1]
stump = fit_stump(xs,ys)
print('gain, threshold, left label, right label:', stump)
print('predictions:', [predict(stump,x) for x in xs])

gain, threshold, left label, right label: (0.5, 3.5, 0, 1)
predictions: [0, 0, 0, 1, 1, 1]


## Bootstrap and out-of-bag rows

Each tree sees a resampled dataset. Rows omitted from its sample are out-of-bag for that tree. Production forests also sample candidate features independently at each split; this one-dimensional example only demonstrates the bootstrap mechanism.

In [3]:
rng = Random(7)
for tree_id in range(3):
    indices = [rng.randrange(len(xs)) for _ in xs]
    oob = sorted(set(range(len(xs))) - set(indices))
    boot_x = [xs[i] for i in indices]
    boot_y = [ys[i] for i in indices]
    tree = fit_stump(boot_x, boot_y)
    print(f'tree {tree_id+1}: threshold={tree[1]}, OOB rows={oob},'
          f' OOB predictions={[(i, predict(tree,xs[i])) for i in oob]}')

tree 1: threshold=3.5, OOB rows=[4], OOB predictions=[(4, 1)]
tree 2: threshold=4.0, OOB rows=[1, 3, 5], OOB predictions=[(1, 0), (3, 0), (5, 1)]
tree 3: threshold=3.0, OOB rows=[2, 4, 5], OOB predictions=[(2, 0), (4, 1), (5, 1)]


### Try it

Change one label and compare training gain to performance on a separate set. Add a second feature and randomly restrict candidate features at each split; examine how votes change.